# Baseline Failures and RG-LNS Improvements on ImageNet-LSEJ

## Analysis Goals

This notebook compares Gallagher, Pomeranz, and LP on the same LSEJ test samples and addresses three questions:

1. Do failures from all three initial solvers exhibit reliable local structures placed at incorrect global positions?
2. Are errors concentrated in low-reliability regions without cycle support?
3. How many failures does RG-LNS repair for each initial solver and failure type?

All population-level proportions use **all failed predictions** from each initial solver as the denominator. Each failure belongs to exactly one category: rigid-misplacement-dominated, low-reliability-region-error-dominated, or other/mixed failure. The three proportions therefore sum to 100%. Incomplete predictions are included in other/mixed failures rather than treated as a separate category in the paper. Qualitative figures are controlled independently by `VISUAL_SOLVER`, which defaults to Gallagher.

## Running This Notebook

Install the optional dependencies and generate paired prediction archives as described in `experiments/README.md`. Run all cells from the repository root or `experiments/`. The distributed notebook has no saved numerical outputs; full-population conclusions require complete test-split archives. Configuration and local input paths are set below.

## Statistical Definitions and Fair Comparison Protocol

### Two Empirical Observations

1. **Consistent internal structure at incorrect global positions.** For piece $i$ in a component, define $\Delta_i=\text{true position}_i-\text{predicted position}_i$. If all pieces share the same nonzero $\Delta_i$, the component preserves its correct internal structure but is displaced by a rigid translation.
2. **Errors concentrated in low-reliability regions.** Adjacency edges are divided into five reliability levels according to their candidate ranks in both directions and support from complete $2\times2$ cycles. We report both $P(\text{error}\mid\text{reliability level})$ and $P(\text{reliability level}\mid\text{error})$.

### Fairness Constraints

- All three initial solvers must use the same tasks, test-sample order, target layouts, metric checkpoint, and RG-LNS structural parameters.
- Baseline metrics use the original `baseline_predictions`; RG-LNS metrics use `selected_predictions` from the same archive for paired per-sample comparisons.
- The current protocol requires `completed_baseline_predictions` and `partial_completion_beam_width` in the archive to verify automatic completion of incomplete initial predictions. Legacy archives are excluded from formal comparisons by default.
- LP may use different metric post-processing settings. Reliability is recomputed using the `postprocess` setting recorded in each RG-LNS run.
- Reliability stratification uses only model scores and predicted permutations. Adjacency correctness, translation consistency, and failure-type labels are used only for offline analysis.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as NotebookImage, Markdown, display
from matplotlib.patches import Rectangle

get_ipython().run_line_magic('matplotlib', 'inline')

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'evaluate.py').is_file() and (candidate / 'compatibility').is_dir():
            return candidate
    raise FileNotFoundError('Cannot locate the RG-LNS repository root; launch Jupyter from the repository')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from compatibility.geometry import BOTTOM, LEFT, OPPOSITE, RIGHT, TOP
from data_loaders.lsej import (
    chw_uint8_to_hwc_numpy, load_official_dataset_class,
    normalize_lsej_task, official_lsej_task, lsej_task_name,
)
from evaluation.lsej import compute_relationship_counts, save_evaluation_visual
from compatibility.scorer import MetricCompatibilityScorer, resolve_device
from paths import COMPATIBILITY_ROOT, LSEJ_DATA_ROOT

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 100)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
def analysis_env(name: str, default: str) -> str:
    """Read RG-LNS analysis settings, accepting earlier environment names."""
    return os.getenv(f'RG_LNS_ANALYSIS_{name}', os.getenv(f'S1A_ANALYSIS_{name}', default))

# For quick checks, set RG_LNS_ANALYSIS_MAX_FAILED to a small positive integer; formal statistics require 0.
TASKS = ('lsej_grid10_erode2',)
INITIAL_SOLVERS = ('Gallagher', 'Pomeranz', 'LP')
VISUAL_SOLVER = analysis_env('VISUAL_SOLVER', 'Gallagher').strip()

# Leave empty for full statistics. A filename selects a single sample and overrides the sample limit.
SELECTED_VISUAL_FILENAME = ''
# SELECTED_VISUAL_FILENAME = 'sample_000179_LSEJ_001123'  # Rigid misplacement example
# SELECTED_VISUAL_FILENAME = 'sample_000315_LSEJ_001951'  # Low-reliability region example
SELECTED_VISUAL_FILENAME = analysis_env('VISUAL_FILENAME', SELECTED_VISUAL_FILENAME).strip()

MAX_FAILED_SAMPLES_PER_TASK = int(analysis_env('MAX_FAILED', '0'))
MUTUAL_TOP_K = 3
MIN_COMPONENT_SIZE = 4
FAILURE_DOMINANCE_THRESHOLD = 0.40
SENSITIVITY_THRESHOLDS = (0.40, 0.50, 0.70, 0.90)
FAILURE_TAXONOMY_VERSION = 'failure_taxonomy_v4_three_initial_solvers'
STRUCTURE_CACHE_VERSION = 'multi_solver_v2'
SAMPLE_NAME_PREVIEW_COUNT = 10
SAMPLE_IMAGE_PREVIEW_COUNT = 0  # Disabled by default; a positive value shows the first M images for VISUAL_SOLVER.
FORCE_RECOMPUTE = analysis_env('FORCE', '0') == '1'
REQUIRE_UNIFIED_PARTIAL_COMPLETION = analysis_env('ALLOW_LEGACY', '0') != '1'
DEVICE = resolve_device('auto', 0)

# Shared defaults come from paths.py; the analysis environment can override the data root.
DATA_ROOT = Path(analysis_env('DATA_ROOT', str(LSEJ_DATA_ROOT))).expanduser().resolve()
RESULT_ROOT = Path(os.getenv(
    'RG_LNS_ANALYSIS_RESULT_ROOT', str(PROJECT_ROOT / 'results' / 'runs' / 'failure_analysis')
)).resolve()
CHECKPOINTS = {
    task: COMPATIBILITY_ROOT / f'{task}_best.pth'
    for task in ('lsej_grid10_erode2', 'lsej_grid10_erode5')
}
SOLVER_COMMANDS = {'Gallagher': 'gallagher', 'Pomeranz': 'pomeranz', 'LP': 'linear-programming'}

if isinstance(TASKS, str):
    TASKS = (TASKS,)
TASKS = tuple(normalize_lsej_task(task) for task in TASKS)
unknown_tasks = set(TASKS) - set(CHECKPOINTS)
if not TASKS or unknown_tasks:
    raise ValueError(f'Choose supported tasks: {tuple(CHECKPOINTS)}; unknown tasks: {sorted(unknown_tasks)}')
if VISUAL_SOLVER not in INITIAL_SOLVERS:
    raise ValueError(f'VISUAL_SOLVER={VISUAL_SOLVER!r} is not in {INITIAL_SOLVERS}')
if MAX_FAILED_SAMPLES_PER_TASK < 0:
    raise ValueError('MAX_FAILED_SAMPLES_PER_TASK must be nonnegative')
LOCAL_DATA_ROOTS = {task: DATA_ROOT for task in TASKS}
LOCAL_CHECKPOINTS = {task: CHECKPOINTS[task] for task in TASKS}


def result_directory(solver: str, task: str) -> Path:
    result_dir = RESULT_ROOT / solver.lower() / task
    if not all((result_dir / name).is_file() for name in ('metrics.json', 'predictions.npz')):
        legacy_dir = RESULT_ROOT / solver.lower() / official_lsej_task(task)
        if all((legacy_dir / name).is_file() for name in ('metrics.json', 'predictions.npz')):
            result_dir = legacy_dir
    if not all((result_dir / name).is_file() for name in ('metrics.json', 'predictions.npz')):
        command = (
            f'python evaluate.py {lsej_task_name(task)} {SOLVER_COMMANDS[solver]} rg-lns '
            f'--split test --checkpoint "{LOCAL_CHECKPOINTS[task]}" --data-root "{DATA_ROOT}" '
            f'--save-predictions true --save-diagnostics true --output-dir "{result_dir}"'
        )
        raise FileNotFoundError(
            f'Missing paired results for {solver} | {task}: {result_dir}\n'
            f'Generate them from the repository root:\n{command}\nSee experiments/README.md.'
        )
    metadata = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
    if (normalize_lsej_task(str(metadata.get('task'))), metadata.get('split'), metadata.get('initial_solver')) != (
        task, 'test', solver
    ):
        raise ValueError(f'{result_dir}: expected {solver} | {task} | test, got incompatible metadata')
    recorded_checkpoint = Path(metadata['checkpoint'])
    if not recorded_checkpoint.is_absolute():
        recorded_checkpoint = PROJECT_ROOT / recorded_checkpoint
    if recorded_checkpoint.resolve() != LOCAL_CHECKPOINTS[task].resolve():
        raise ValueError(
            f'{result_dir}: checkpoint differs from CHECKPOINTS[{task!r}]. '
            'Update CHECKPOINTS to the exact checkpoint used by these archives.'
        )
    issues = []
    configuration = metadata.get('configuration', {})
    if 'partial_completion_beam_width' not in configuration:
        issues.append('metrics.json is missing partial_completion_beam_width')
    with np.load(result_dir / 'predictions.npz') as archive:
        required = {
            'baseline_predictions', 'selected_predictions', 'targets', 'lsej_ids', 'sample_indices'
        }
        missing = required - set(archive.files)
        if missing:
            raise ValueError(f'{result_dir}: missing prediction fields: {sorted(missing)}')
        if 'completed_baseline_predictions' not in archive.files:
            issues.append('predictions.npz is missing completed_baseline_predictions')
        if not SELECTED_VISUAL_FILENAME and MAX_FAILED_SAMPLES_PER_TASK == 0:
            dataset_class = load_official_dataset_class(DATA_ROOT)
            dataset = dataset_class(DATA_ROOT, split='test', task=official_lsej_task(task), normalize=False)
            if not np.array_equal(np.sort(archive['sample_indices']), np.arange(len(dataset))):
                raise ValueError(
                    f'{result_dir}: full-population mode requires every test sample exactly once. '
                    'Generate a full run, or set RG_LNS_ANALYSIS_MAX_FAILED to a positive value for a quick check.'
                )
    if issues and REQUIRE_UNIFIED_PARTIAL_COMPLETION:
        raise ValueError(f'{result_dir}: {"; ".join(issues)}. Regenerate this run with the public evaluator.')
    if issues:
        print(f'Warning: legacy archive for workflow checks only: {result_dir}')
    return result_dir


RG_LNS_RESULT_DIRS = {
    (solver, task): result_directory(solver, task)
    for solver in INITIAL_SOLVERS for task in TASKS
}
for task in TASKS:
    if not LOCAL_CHECKPOINTS[task].is_file():
        raise FileNotFoundError(f'Missing checkpoint: {LOCAL_CHECKPOINTS[task]}')
CACHE_DIR = RESULT_ROOT / 'analysis_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

selected_input = str(SELECTED_VISUAL_FILENAME).strip().strip(chr(34) + chr(39) + '`')
if selected_input:
    filename_match = re.search(r'(?i)sample[_-](\d+)[_-](LSEJ[_-]\d+)(?:\.png)?', selected_input)
    if filename_match is None:
        raise ValueError(
            'Cannot parse the selected sample. Use a full path or filename with or without .png, such as '
            f'sample_000014_LSEJ_000097.png. Current input: {SELECTED_VISUAL_FILENAME!r}'
        )
    SELECTED_DATASET_INDEX = int(filename_match.group(1))
    SELECTED_LSEJ_ID = filename_match.group(2).upper().replace('-', '_')
    selected_filename = f'sample_{SELECTED_DATASET_INDEX:06d}_{SELECTED_LSEJ_ID}.png'
    selection_label = f'file_{Path(selected_filename).stem}'
    selection_mode = f'Selected file: {selected_filename} (sample limit ignored)'
else:
    selected_filename = ''
    SELECTED_DATASET_INDEX = None
    SELECTED_LSEJ_ID = None
    limit_label = 'all' if MAX_FAILED_SAMPLES_PER_TASK == 0 else str(MAX_FAILED_SAMPLES_PER_TASK)
    selection_label = f'complete_failed_n{limit_label}'
    selection_mode = f'Analyze {MAX_FAILED_SAMPLES_PER_TASK or "all"} complete failed samples per solver/task'

task_label = '-'.join(TASKS)
# Invalidate cached scores when an archive, checkpoint, or local data path changes.
cache_inputs = [str(DATA_ROOT.resolve()), str(REQUIRE_UNIFIED_PARTIAL_COMPLETION)]
for result_dir in RG_LNS_RESULT_DIRS.values():
    for name in ('metrics.json', 'predictions.npz'):
        path = result_dir / name
        cache_inputs.append(f'{path.resolve()}:{path.stat().st_size}:{path.stat().st_mtime_ns}')
for path in LOCAL_CHECKPOINTS.values():
    cache_inputs.append(f'{path.resolve()}:{path.stat().st_size}:{path.stat().st_mtime_ns}')
run_label = hashlib.sha256('|'.join(cache_inputs).encode()).hexdigest()[:16]
CACHE_TAG = (
    f'{task_label}_{selection_label}_k{MUTUAL_TOP_K}_min{MIN_COMPONENT_SIZE}_'
    f'{STRUCTURE_CACHE_VERSION}_{run_label}'
)
print({
    'Device': str(DEVICE), 'Initial solver': INITIAL_SOLVERS, 'Qualitative solver': VISUAL_SOLVER,
    'Sample selection mode': selection_mode, 'Mutual Top-K': MUTUAL_TOP_K,
    'Failure dominance threshold': f'{FAILURE_DOMINANCE_THRESHOLD:.0%}',
    'Strict unified completion protocol': REQUIRE_UNIFIED_PARTIAL_COMPLETION,
    'Baseline + RG-LNS results': {f'{s} | {t}': str(p) for (s, t), p in RG_LNS_RESULT_DIRS.items()},
})

## Data Sources and Protocol Audit

For each initial solver, the notebook loads initial predictions, RG-LNS predictions, target positions, and sample identifiers saved together in the corresponding run directory. It first verifies identical sample order and target layouts, then compares initial and profiled results across the three solvers. If the latest archive predates automatic completion, strict mode stops and requests a new run to prevent incomparable results from entering the paper tables.

In [ ]:
DIRECTION_OFFSETS = {TOP: (-1, 0), RIGHT: (0, 1), BOTTOM: (1, 0), LEFT: (0, -1)}
BUCKET_ORDER = [
    'mutual Top-1 + cycle',
    f'mutual Top-2..{MUTUAL_TOP_K} + cycle',
    f'mutual Top-{MUTUAL_TOP_K}, no cycle',
    f'one-way Top-{MUTUAL_TOP_K}',
    f'neither Top-{MUTUAL_TOP_K}',
]
BUCKET_COLORS = ['#087f5b', '#40c057', '#fab005', '#fd7e14', '#e03131']
BUCKET_LABELS = {
    BUCKET_ORDER[0]: 'Mutual Top-1 + cycle support',
    BUCKET_ORDER[1]: f'Mutual Top-{MUTUAL_TOP_K} + cycle support',
    BUCKET_ORDER[2]: f'Mutual Top-{MUTUAL_TOP_K}, no cycle support',
    BUCKET_ORDER[3]: f'One-way Top-{MUTUAL_TOP_K}',
    BUCKET_ORDER[4]: f'Neither direction in Top-{MUTUAL_TOP_K}',
}

def is_complete_positions(positions: np.ndarray, grid: int) -> bool:
    positions = np.asarray(positions, dtype=np.int64)
    expected = np.arange(grid * grid, dtype=np.int64)
    return positions.shape == expected.shape and np.array_equal(np.sort(positions), expected)

def positions_to_layout(positions: np.ndarray, grid: int) -> np.ndarray:
    if not is_complete_positions(positions, grid):
        raise ValueError('Adjacency and component analysis requires a complete permutation of 0..grid²-1')
    layout = np.empty(grid * grid, dtype=np.int32)
    layout[np.asarray(positions, dtype=np.int64)] = np.arange(grid * grid, dtype=np.int32)
    return layout.reshape(grid, grid)

def candidate_rank(scores: np.ndarray, anchor: int, candidate: int, direction: int) -> int:
    candidate_score = float(scores[anchor, candidate, direction])
    mask = np.ones(scores.shape[0], dtype=bool)
    mask[anchor] = False
    return int(np.count_nonzero(mask & (scores[anchor, :, direction] <= candidate_score)))

def edge_key(first: int, second: int) -> tuple[int, int]:
    return tuple(sorted((int(first), int(second))))

def current_edge_records(prediction: np.ndarray, target: np.ndarray, scores: np.ndarray, grid: int):
    layout = positions_to_layout(prediction, grid)
    records = []
    mutual_edges = set()

    for row in range(grid):
        for column in range(grid):
            anchor = int(layout[row, column])
            for direction, dr, dc in ((RIGHT, 0, 1), (BOTTOM, 1, 0)):
                nr, nc = row + dr, column + dc
                if nr >= grid or nc >= grid:
                    continue
                candidate = int(layout[nr, nc])
                reverse = int(OPPOSITE[direction])
                forward_rank = candidate_rank(scores, anchor, candidate, direction)
                reverse_rank = candidate_rank(scores, candidate, anchor, reverse)
                is_mutual = max(forward_rank, reverse_rank) <= MUTUAL_TOP_K
                if is_mutual:
                    mutual_edges.add(edge_key(anchor, candidate))
                target_anchor = divmod(int(target[anchor]), grid)
                target_candidate = divmod(int(target[candidate]), grid)
                expected = DIRECTION_OFFSETS[direction]
                is_correct = (
                    target_candidate[0] - target_anchor[0] == expected[0]
                    and target_candidate[1] - target_anchor[1] == expected[1]
                )
                records.append({
                    'row': row, 'column': column, 'direction': int(direction),
                    'anchor': anchor, 'candidate': candidate,
                    'forward_rank': forward_rank, 'reverse_rank': reverse_rank,
                    'worst_rank': max(forward_rank, reverse_rank),
                    'one_way_top_k': min(forward_rank, reverse_rank) <= MUTUAL_TOP_K,
                    'mutual_top_k': is_mutual,
                    'score': float(scores[anchor, candidate, direction]),
                    'correct': bool(is_correct),
                    'edge_key': edge_key(anchor, candidate),
                })

    supported_edges = set()
    for row in range(grid - 1):
        for column in range(grid - 1):
            tl, tr = int(layout[row, column]), int(layout[row, column + 1])
            bl, br = int(layout[row + 1, column]), int(layout[row + 1, column + 1])
            perimeter = {edge_key(tl, tr), edge_key(tl, bl), edge_key(tr, br), edge_key(bl, br)}
            if perimeter <= mutual_edges:
                supported_edges.update(perimeter)

    for record in records:
        record['cycle_supported'] = record['edge_key'] in supported_edges
        if record['cycle_supported'] and record['worst_rank'] == 1:
            bucket = BUCKET_ORDER[0]
        elif record['cycle_supported']:
            bucket = BUCKET_ORDER[1]
        elif record['mutual_top_k']:
            bucket = BUCKET_ORDER[2]
        elif record['one_way_top_k']:
            bucket = BUCKET_ORDER[3]
        else:
            bucket = BUCKET_ORDER[4]
        record['reliability_bucket'] = bucket
    return records, supported_edges

def supported_components(supported_edges: set[tuple[int, int]], num_pieces: int):
    parents = list(range(num_pieces))
    def find(piece):
        while parents[piece] != piece:
            parents[piece] = parents[parents[piece]]
            piece = parents[piece]
        return piece
    def union(first, second):
        first_root, second_root = find(first), find(second)
        if first_root != second_root:
            parents[second_root] = first_root
    supported_pieces = set()
    for first, second in supported_edges:
        union(first, second)
        supported_pieces.update((first, second))
    groups = {}
    for piece in supported_pieces:
        groups.setdefault(find(piece), []).append(piece)
    components = [sorted(group) for group in groups.values() if len(group) >= MIN_COMPONENT_SIZE]
    return sorted(components, key=lambda group: (-len(group), min(group)))

def component_record(component, prediction, target, edge_records, component_index, grid):
    component_set = set(component)
    deltas = []
    for piece in component:
        predicted_row, predicted_column = divmod(int(prediction[piece]), grid)
        target_row, target_column = divmod(int(target[piece]), grid)
        deltas.append((target_row - predicted_row, target_column - predicted_column))
    dominant_shift, dominant_count = Counter(deltas).most_common(1)[0]
    internal_edges = [
        edge for edge in edge_records
        if edge['anchor'] in component_set and edge['candidate'] in component_set
    ]
    supported_internal = [edge for edge in internal_edges if edge['cycle_supported']]
    return {
        'component_index': component_index,
        'size': len(component),
        'piece_ids': ','.join(map(str, component)),
        'translation_consistency': dominant_count / len(component),
        'dominant_row_shift': dominant_shift[0],
        'dominant_column_shift': dominant_shift[1],
        'nonzero_dominant_shift': dominant_shift != (0, 0),
        'perfect_rigid_translation': dominant_count == len(component),
        'coherent_misplacement': dominant_count == len(component) and dominant_shift != (0, 0),
        'absolute_piece_accuracy': float(np.mean(np.asarray(prediction)[component] == np.asarray(target)[component])),
        'internal_edges': len(internal_edges),
        'internal_edge_accuracy': float(np.mean([edge['correct'] for edge in internal_edges])) if internal_edges else np.nan,
        'supported_internal_edges': len(supported_internal),
        'supported_edge_accuracy': float(np.mean([edge['correct'] for edge in supported_internal])) if supported_internal else np.nan,
    }

def analyze_task(task: str, initial_solver: str, stage: str = 'Baseline'):
    key = (initial_solver, task)
    result_dir = RG_LNS_RESULT_DIRS[key]
    run_metadata = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
    prediction_key = {'Baseline': 'baseline_predictions', 'Baseline + RG-LNS': 'selected_predictions'}.get(stage)
    if prediction_key is None:
        raise ValueError(f'Unknown analysis stage: {stage}')
    with np.load(result_dir / 'predictions.npz') as archive:
        predictions = archive[prediction_key].copy()
        targets = archive['targets'].copy()
        lsej_ids = archive['lsej_ids'].astype(str)
        dataset_indices = archive['sample_indices'].astype(np.int64)
    grid = int(round(np.sqrt(predictions.shape[1])))
    complete_mask = np.asarray([is_complete_positions(row, grid) for row in predictions], dtype=bool)
    failed_mask = np.any(predictions != targets, axis=1)
    complete_failed_rows = np.flatnonzero(failed_mask & complete_mask)
    print(
        f'{initial_solver} | {task} | {stage}: perfect layouts={int(np.count_nonzero(~failed_mask)):,}, '
        f'complete failed layouts={len(complete_failed_rows):,}, incomplete predictions={int(np.count_nonzero(~complete_mask)):,}'
    )
    if selected_filename:
        matching_rows = np.flatnonzero(dataset_indices == int(SELECTED_DATASET_INDEX))
        if len(matching_rows) != 1:
            raise IndexError(f'The dataset index in {selected_filename} is absent from the {initial_solver} | {task} results')
        saved_row = int(matching_rows[0])
        if lsej_ids[saved_row] != SELECTED_LSEJ_ID:
            raise ValueError(
                f'{initial_solver} | {task}: index {SELECTED_DATASET_INDEX} corresponds to {lsej_ids[saved_row]}, '
                f'but the selected filename contains {SELECTED_LSEJ_ID}'
            )
        if not complete_mask[saved_row]:
            print(f'Skipping the selected sample for {initial_solver} | {stage}: prediction is not a complete permutation.')
            selected_rows = np.asarray([], dtype=np.int64)
        else:
            selected_rows = np.asarray([saved_row], dtype=np.int64)
    else:
        selected_rows = complete_failed_rows
        if MAX_FAILED_SAMPLES_PER_TASK:
            selected_rows = selected_rows[:MAX_FAILED_SAMPLES_PER_TASK]

    dataset_class = load_official_dataset_class(LOCAL_DATA_ROOTS[task])
    dataset = dataset_class(
        LOCAL_DATA_ROOTS[task], split=run_metadata['split'], task=official_lsej_task(task), normalize=False
    )
    postprocess = bool(run_metadata.get('configuration', {}).get('postprocess', True))
    scorer = MetricCompatibilityScorer.from_checkpoint(
        LOCAL_CHECKPOINTS[task], device=DEVICE, batch_size=128, postprocess=postprocess
    )
    edge_rows, component_rows, sample_rows = [], [], []
    for progress, saved_row in enumerate(selected_rows, start=1):
        dataset_index = int(dataset_indices[saved_row])
        sample = dataset[dataset_index]
        if str(sample['lsej_id']) != lsej_ids[saved_row]:
            raise RuntimeError(f'{initial_solver} | {task}: the LSEJ ID at row {saved_row} does not match the dataset')
        pieces = list(chw_uint8_to_hwc_numpy(sample['pieces']))
        scores = scorer.score_pieces(pieces, rot_flag=0)
        prediction, target = predictions[saved_row], targets[saved_row]
        records, supported_edges = current_edge_records(prediction, target, scores, grid)
        current_components_raw = supported_components(supported_edges, grid * grid)
        for record in records:
            record.pop('edge_key')
            record.update(
                initial_solver=initial_solver, stage=stage, task=task,
                lsej_id=lsej_ids[saved_row], saved_row=int(saved_row),
            )
            edge_rows.append(record)
        current_components = []
        for component_index, component in enumerate(current_components_raw):
            record = component_record(component, prediction, target, records, component_index, grid)
            record.update(
                initial_solver=initial_solver, stage=stage, task=task,
                lsej_id=lsej_ids[saved_row], saved_row=int(saved_row),
            )
            component_rows.append(record)
            current_components.append(record)
        wrong_pieces = set(np.flatnonzero(np.asarray(prediction) != np.asarray(target)).tolist())
        reliable_pieces = {piece for component in current_components_raw for piece in component}
        rigid_misplacement_pieces = {
            piece
            for component, component_info in zip(current_components_raw, current_components)
            if component_info['coherent_misplacement']
            for piece in component
        }
        low_reliability_pieces = set(range(grid * grid)) - reliable_pieces
        incorrect_records = [record for record in records if not record['correct']]
        low_reliability_incorrect_records = [
            record for record in incorrect_records if not record['cycle_supported']
        ]
        wrong_piece_count = len(wrong_pieces)
        rigid_wrong_piece_count = len(wrong_pieces & rigid_misplacement_pieces)
        low_reliability_wrong_piece_count = len(wrong_pieces & low_reliability_pieces)
        sample_rows.append({
            'initial_solver': initial_solver, 'stage': stage, 'task': task,
            'lsej_id': lsej_ids[saved_row], 'saved_row': int(saved_row),
            'dataset_index': dataset_index,
            'is_perfect_layout': bool(np.array_equal(prediction, target)),
            'reliable_components': len(current_components),
            'largest_component_size': max((row['size'] for row in current_components), default=0),
            'has_reliable_component': bool(current_components),
            'has_coherent_misplacement': any(row['coherent_misplacement'] for row in current_components),
            'wrong_pieces': wrong_piece_count,
            'rigid_misplacement_pieces': len(rigid_misplacement_pieces),
            'rigid_wrong_pieces': rigid_wrong_piece_count,
            'rigid_wrong_piece_ratio': rigid_wrong_piece_count / wrong_piece_count if wrong_piece_count else np.nan,
            'low_reliability_pieces': len(low_reliability_pieces),
            'low_reliability_wrong_pieces': low_reliability_wrong_piece_count,
            'low_reliability_wrong_piece_ratio': low_reliability_wrong_piece_count / wrong_piece_count if wrong_piece_count else np.nan,
            'incorrect_adjacencies': len(incorrect_records),
            'low_reliability_incorrect_adjacencies': len(low_reliability_incorrect_records),
            'low_reliability_incorrect_adjacency_ratio': (
                len(low_reliability_incorrect_records) / len(incorrect_records)
                if incorrect_records else np.nan
            ),
        })
        if progress == 1 or progress % 25 == 0 or progress == len(selected_rows):
            print(f'{initial_solver} | {task} | {stage}: processed {progress}/{len(selected_rows)} samples')
    columns = {
        'edges': ['initial_solver', 'stage', 'task', 'lsej_id', 'saved_row'],
        'components': ['initial_solver', 'stage', 'task', 'lsej_id', 'saved_row'],
        'samples': ['initial_solver', 'stage', 'task', 'lsej_id', 'saved_row'],
    }
    frames = [pd.DataFrame(edge_rows), pd.DataFrame(component_rows), pd.DataFrame(sample_rows)]
    for frame, kind in zip(frames, ('edges', 'components', 'samples')):
        if frame.empty:
            for column in columns[kind]:
                frame[column] = pd.Series(dtype='object')
    return tuple(frames)


In [ ]:
# Read paired per-sample archives for all three solvers and verify identical test samples and target layouts.
population_audit_rows = []
overall_comparison_rows = []
paired_outcome_frames = []
RG_LNS_BASELINE_PREDICTIONS = {}
RG_LNS_SELECTED_PREDICTIONS = {}
RG_LNS_TARGETS = {}
RG_LNS_LSEJ_IDS = {}
RG_LNS_SAMPLE_INDICES = {}
RUN_METADATA = {}

def aggregate_layout_metrics(predictions: np.ndarray, targets: np.ndarray, grid: int) -> dict:
    correct_pieces = int(np.count_nonzero(predictions == targets))
    perfect = int(np.count_nonzero(np.all(predictions == targets, axis=1)))
    horizontal_correct = horizontal_total = vertical_correct = vertical_total = 0
    for prediction, target in zip(predictions, targets):
        counts = compute_relationship_counts(prediction, target, grid)
        horizontal_correct += int(counts['horizontal_correct'])
        horizontal_total += int(counts['horizontal_total'])
        vertical_correct += int(counts['vertical_correct'])
        vertical_total += int(counts['vertical_total'])
    return {
        'Perfect layouts': perfect,
        'PA': perfect / len(predictions),
        'AA': correct_pieces / predictions.size,
        'SRA': (horizontal_correct + vertical_correct) / (horizontal_total + vertical_total),
        'Incomplete predictions': int(np.count_nonzero([
            not is_complete_positions(row, grid) for row in predictions
        ])),
    }

reference_arrays_by_task = {}
checkpoint_name_by_task = {}
structural_keys = (
    'completion_beam_width', 'partial_completion_beam_width', 'mutual_top_k',
    'component_index', 'min_component_size', 'refinement_rounds',
)
protocol_rows = []
reference_structure = None
for initial_solver in INITIAL_SOLVERS:
    for task in TASKS:
        key = (initial_solver, task)
        result_dir = RG_LNS_RESULT_DIRS[key]
        metadata = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
        RUN_METADATA[key] = metadata
        with np.load(result_dir / 'predictions.npz') as archive:
            baseline = archive['baseline_predictions'].copy()
            selected = archive['selected_predictions'].copy()
            targets = archive['targets'].copy()
            lsej_ids = archive['lsej_ids'].astype(str)
            sample_indices = archive['sample_indices'].astype(np.int64)
            archive_keys = set(archive.files)
        if not (len(baseline) == len(selected) == len(targets) == len(lsej_ids) == len(sample_indices)):
            raise RuntimeError(f'{initial_solver} | {task}: paired archives have inconsistent lengths')
        if task not in reference_arrays_by_task:
            reference_arrays_by_task[task] = (targets, lsej_ids, sample_indices)
        else:
            ref_targets, ref_ids, ref_indices = reference_arrays_by_task[task]
            if not (
                np.array_equal(targets, ref_targets)
                and np.array_equal(lsej_ids, ref_ids)
                and np.array_equal(sample_indices, ref_indices)
            ):
                raise RuntimeError(f'{initial_solver} | {task}: sample order or target layouts differ from the other solvers')
        checkpoint_path = Path(str(metadata.get('checkpoint', '')))
        checkpoint_identity = str(checkpoint_path.resolve())
        if task not in checkpoint_name_by_task:
            checkpoint_name_by_task[task] = checkpoint_identity
        elif checkpoint_identity != checkpoint_name_by_task[task]:
            raise RuntimeError(f'{initial_solver} | {task} uses a different metric checkpoint: {checkpoint_identity}')
        configuration = metadata.get('configuration', {})
        structure_signature = tuple(configuration.get(name) for name in structural_keys)
        if reference_structure is None:
            reference_structure = structure_signature
        elif REQUIRE_UNIFIED_PARTIAL_COMPLETION and structure_signature != reference_structure:
            raise RuntimeError(
                f'{initial_solver} | {task}: RG-LNS structural parameters differ from the first archive: '
                f'{dict(zip(structural_keys, structure_signature))}'
            )
        unified = (
            'completed_baseline_predictions' in archive_keys
            and 'partial_completion_beam_width' in configuration
        )
        protocol_rows.append({
            'Initial solver': initial_solver, 'Task': task, 'Test samples': len(targets),
            'Unified automatic completion protocol': unified, 'checkpoint': checkpoint_identity,
            'postprocess': bool(configuration.get('postprocess', True)),
            **{name: configuration.get(name) for name in structural_keys},
        })
        grid = int(round(np.sqrt(baseline.shape[1])))
        baseline_complete = np.asarray([is_complete_positions(row, grid) for row in baseline], dtype=bool)
        selected_complete = np.asarray([is_complete_positions(row, grid) for row in selected], dtype=bool)
        baseline_failed = np.any(baseline != targets, axis=1)
        selected_failed = np.any(selected != targets, axis=1)
        RG_LNS_BASELINE_PREDICTIONS[key] = baseline
        RG_LNS_SELECTED_PREDICTIONS[key] = selected
        RG_LNS_TARGETS[key] = targets
        RG_LNS_LSEJ_IDS[key] = lsej_ids
        RG_LNS_SAMPLE_INDICES[key] = sample_indices
        population_audit_rows.append({
            'Initial solver': initial_solver, 'Task': task, 'Total predictions': len(baseline),
            'Baseline perfect layouts': int(np.count_nonzero(~baseline_failed)),
            'Baseline complete failures': int(np.count_nonzero(baseline_complete & baseline_failed)),
            'Baseline incomplete predictions': int(np.count_nonzero(~baseline_complete)),
            'Baseline + RG-LNS perfect layouts': int(np.count_nonzero(~selected_failed)),
            'Baseline + RG-LNS complete failures': int(np.count_nonzero(selected_complete & selected_failed)),
            'Baseline + RG-LNS incomplete predictions': int(np.count_nonzero(~selected_complete)),
        })
        paired_outcome_frames.append(pd.DataFrame({
            'initial_solver': initial_solver, 'task': task, 'lsej_id': lsej_ids,
            'saved_row': np.arange(len(baseline), dtype=np.int64),
            'dataset_index': sample_indices,
            'baseline_complete': baseline_complete, 'ours_complete': selected_complete,
            'baseline_perfect': ~baseline_failed, 'ours_perfect': ~selected_failed,
            'baseline_wrong_pieces': np.count_nonzero(baseline != targets, axis=1),
            'ours_wrong_pieces': np.count_nonzero(selected != targets, axis=1),
            'prediction_changed': np.any(baseline != selected, axis=1),
        }))
        for stage, predictions in (('Baseline', baseline), ('Baseline + RG-LNS', selected)):
            metric_row = aggregate_layout_metrics(predictions, targets, grid)
            metric_row.update({'Initial solver': initial_solver, 'Task': task, 'Stage': stage})
            overall_comparison_rows.append(metric_row)

protocol_audit = pd.DataFrame(protocol_rows).set_index(['Initial solver', 'Task'])
population_audit = pd.DataFrame(population_audit_rows).set_index(['Initial solver', 'Task'])
paired_outcomes = pd.concat(paired_outcome_frames, ignore_index=True)
display(Markdown('### Protocol Consistency Audit'))
display(protocol_audit)
display(Markdown('### Prediction Population Audit'))
display(population_audit)
display(Markdown('### Overall Comparison of Three Initial Solvers and Baseline + RG-LNS'))
overall_comparison = pd.DataFrame(overall_comparison_rows).set_index(['Initial solver', 'Task', 'Stage'])
display(overall_comparison.style.format({'PA': '{:.1%}', 'AA': '{:.1%}', 'SRA': '{:.1%}'}))

def load_or_analyze(initial_solver: str, stage: str):
    solver_slug = initial_solver.lower()
    stage_slug = {'Baseline': 'baseline', 'Baseline + RG-LNS': 'rg_lns'}[stage]
    cache_prefix = f'{solver_slug}_{stage_slug}_{CACHE_TAG}'
    paths = (
        CACHE_DIR / f'edges_{cache_prefix}.csv',
        CACHE_DIR / f'components_{cache_prefix}.csv',
        CACHE_DIR / f'samples_{cache_prefix}.csv',
    )
    if not FORCE_RECOMPUTE and all(path.is_file() for path in paths):
        frames = tuple(pd.read_csv(path) for path in paths)
        print(f'Loaded cache: {initial_solver} | {stage}')
    else:
        outputs = [analyze_task(task, initial_solver, stage) for task in TASKS]
        frames = tuple(pd.concat([output[index] for output in outputs], ignore_index=True) for index in range(3))
        for frame, path in zip(frames, paths):
            frame.to_csv(path, index=False)
        print(f'Saved cache: {initial_solver} | {stage}')
    for frame in frames:
        if 'initial_solver' not in frame.columns:
            frame['initial_solver'] = initial_solver
        if 'stage' not in frame.columns:
            frame['stage'] = stage
    return frames

initial_outputs = [load_or_analyze(solver, 'Baseline') for solver in INITIAL_SOLVERS]
ours_outputs = [load_or_analyze(solver, 'Baseline + RG-LNS') for solver in INITIAL_SOLVERS]
edges = pd.concat([output[0] for output in initial_outputs], ignore_index=True)
components = pd.concat([output[1] for output in initial_outputs], ignore_index=True)
samples = pd.concat([output[2] for output in initial_outputs], ignore_index=True)
ours_edges = pd.concat([output[0] for output in ours_outputs], ignore_index=True)
ours_components = pd.concat([output[1] for output in ours_outputs], ignore_index=True)
ours_samples = pd.concat([output[2] for output in ours_outputs], ignore_index=True)
print(f'Baseline: adjacency edges={len(edges):,}, components={len(components):,}, samples={len(samples):,}')
print(f'Baseline + RG-LNS: adjacency edges={len(ours_edges):,}, components={len(ours_components):,}, samples={len(ours_samples):,}')

if SAMPLE_NAME_PREVIEW_COUNT < 0 or SAMPLE_IMAGE_PREVIEW_COUNT < 0:
    raise ValueError('Preview counts must be nonnegative')
visual_samples = samples[samples['initial_solver'] == VISUAL_SOLVER]
if SAMPLE_NAME_PREVIEW_COUNT > 0 and not visual_samples.empty:
    preview = visual_samples.groupby('task', sort=False).head(SAMPLE_NAME_PREVIEW_COUNT).copy()
    preview['Visualization filename'] = preview.apply(
        lambda row: f"sample_{int(row['dataset_index']):06d}_{row['lsej_id']}.png", axis=1
    )
    display(preview[[
        'initial_solver', 'task', 'lsej_id', 'saved_row', 'dataset_index', 'Visualization filename'
    ]].rename(columns={
        'initial_solver': 'Initial solver', 'task': 'Task', 'lsej_id': 'Sample name',
        'saved_row': 'Result row', 'dataset_index': 'Dataset index',
    }).reset_index(drop=True))

if SAMPLE_IMAGE_PREVIEW_COUNT > 0 and not visual_samples.empty:
    generated_root = CACHE_DIR / 'visual_previews' / CACHE_TAG
    for task, task_samples in visual_samples.groupby('task', sort=False):
        dataset_class = load_official_dataset_class(LOCAL_DATA_ROOTS[task])
        key = (VISUAL_SOLVER, task)
        metadata = RUN_METADATA[key]
        dataset = dataset_class(LOCAL_DATA_ROOTS[task], split=metadata['split'], task=task, normalize=False)
        for row in task_samples.head(SAMPLE_IMAGE_PREVIEW_COUNT).itertuples(index=False):
            saved_row = int(row.saved_row)
            filename = f'sample_{int(row.dataset_index):06d}_{row.lsej_id}.png'
            visual_path = generated_root / VISUAL_SOLVER.lower() / task / filename
            if not visual_path.is_file():
                sample = dataset[int(row.dataset_index)]
                save_evaluation_visual(
                    visual_path, list(chw_uint8_to_hwc_numpy(sample['pieces'])),
                    RG_LNS_BASELINE_PREDICTIONS[key][saved_row], RG_LNS_TARGETS[key][saved_row],
                    int(round(np.sqrt(RG_LNS_TARGETS[key].shape[1]))),
                    f'{VISUAL_SOLVER} | {task} | {row.lsej_id}', method_name=VISUAL_SOLVER,
                )
            display(Markdown(f'#### {VISUAL_SOLVER} | {task} | {filename}'))
            display(NotebookImage(filename=str(visual_path), width=1200))

In [ ]:
# Validate row counts, value ranges, and per-sample pairing before interpreting derived results.
def validate_structure_frames(edge_frame: pd.DataFrame, sample_frame: pd.DataFrame, stage: str):
    if sample_frame.empty:
        return
    expected = sample_frame.groupby(['initial_solver', 'task']).size() * 2 * 10 * 9
    observed = edge_frame.groupby(['initial_solver', 'task']).size()
    assert observed.equals(expected), (stage, observed, expected)
    assert set(edge_frame['reliability_bucket'].unique()) <= set(BUCKET_ORDER)
    assert edge_frame[['forward_rank', 'reverse_rank']].min().min() >= 1
    assert edge_frame[['correct', 'cycle_supported', 'mutual_top_k']].notna().all().all()

validate_structure_frames(edges, samples, 'Baseline')
validate_structure_frames(ours_edges, ours_samples, 'Baseline + RG-LNS')
assert components.empty or (components['size'] >= MIN_COMPONENT_SIZE).all()
assert ours_components.empty or (ours_components['size'] >= MIN_COMPONENT_SIZE).all()
assert not paired_outcomes.duplicated(['initial_solver', 'task', 'saved_row']).any()
assert (population_audit['Baseline perfect layouts'] + population_audit['Baseline complete failures'] + population_audit['Baseline incomplete predictions'] == population_audit['Total predictions']).all()
assert (population_audit['Baseline + RG-LNS perfect layouts'] + population_audit['Baseline + RG-LNS complete failures'] + population_audit['Baseline + RG-LNS incomplete predictions'] == population_audit['Total predictions']).all()
print('Quality checks passed: per-sample archives, adjacency counts, and derived labels agree across all three solvers.')

## Mutually Exclusive Classification of All Failed Predictions

For each initial solver, the population consists of **all failed predictions**. For a complete failed layout, let $W_s$ be the set of pieces at incorrect absolute positions:

1. **Rigid-misplacement-dominated:** at least a fraction $\tau$ of $|W_s|$ belongs to rigidly misplaced components.
2. **Low-reliability-region-error-dominated:** the sample does not meet the first criterion, and at least a fraction $\tau$ of $|W_s|$ is not covered by any cycle-supported reliable component.
3. **Other/mixed failure:** all remaining complete failures and incomplete predictions for which full-grid adjacency cannot be defined.

The main analysis uses $\tau=40\%$. Applying the rules in this order makes the categories mutually exclusive and exhaustive. The table reports the initial failure composition, which sums to 100% for each solver, the residual failure composition after RG-LNS, and repair rates by category. Quick checks and single-sample mode are used only to inspect the workflow and do not support population-level paper conclusions.

In [ ]:
FAILURE_CLASS_ORDER = ['Rigid-misplacement-dominated', 'Low-reliability-region-error-dominated', 'Other/mixed failure']
FAILURE_CLASS_COLORS = {
    'Rigid-misplacement-dominated': '#228be6', 'Low-reliability-region-error-dominated': '#fd7e14', 'Other/mixed failure': '#868e96',
}

def assign_failure_type(row: pd.Series, threshold: float) -> str:
    if bool(row.get('is_perfect_layout', False)):
        return 'Perfect layout (not a failure)'
    if float(row['rigid_wrong_piece_ratio']) >= threshold:
        return FAILURE_CLASS_ORDER[0]
    if float(row['low_reliability_wrong_piece_ratio']) >= threshold:
        return FAILURE_CLASS_ORDER[1]
    return FAILURE_CLASS_ORDER[2]

failure_samples = samples.copy()
ours_failure_samples = ours_samples.copy()
if not failure_samples.empty:
    failure_samples['failure_type'] = failure_samples.apply(
        assign_failure_type, axis=1, threshold=FAILURE_DOMINANCE_THRESHOLD
    )
if not ours_failure_samples.empty:
    ours_failure_samples['failure_type'] = ours_failure_samples.apply(
        assign_failure_type, axis=1, threshold=FAILURE_DOMINANCE_THRESHOLD
    )

FULL_POPULATION_ANALYSIS = (not selected_filename and MAX_FAILED_SAMPLES_PER_TASK == 0)

def classify_all_failures(outcomes: pd.DataFrame, complete_samples: pd.DataFrame, stage: str):
    perfect_column = 'baseline_perfect' if stage == 'Baseline' else 'ours_perfect'
    complete_column = 'baseline_complete' if stage == 'Baseline' else 'ours_complete'
    failed = outcomes[~outcomes[perfect_column].astype(bool)].copy()
    lookup = complete_samples[[
        'initial_solver', 'task', 'saved_row', 'failure_type'
    ]].drop_duplicates(['initial_solver', 'task', 'saved_row'])
    failed = failed.merge(
        lookup, on=['initial_solver', 'task', 'saved_row'], how='left', validate='one_to_one'
    )
    failed.loc[~failed[complete_column].astype(bool), 'failure_type'] = FAILURE_CLASS_ORDER[2]
    if failed['failure_type'].isna().any():
        missing = failed.loc[failed['failure_type'].isna(), ['initial_solver', 'task', 'saved_row']].head()
        raise RuntimeError(f'{stage}: complete failed samples lack structural labels; rerun in full-population mode:\n{missing}')
    failed['stage'] = stage
    return failed

if FULL_POPULATION_ANALYSIS:
    classified_initial = classify_all_failures(paired_outcomes, failure_samples, 'Baseline')
    classified_ours = classify_all_failures(paired_outcomes, ours_failure_samples, 'Baseline + RG-LNS')
    classified_outcomes = classified_initial.copy()
    classified_outcomes['repaired_to_perfect'] = classified_outcomes['ours_perfect'].astype(bool)
    all_classified = pd.concat([classified_initial, classified_ours], ignore_index=True)

    counts = (
        all_classified.groupby(['initial_solver', 'stage', 'failure_type']).size()
        .unstack(fill_value=0).reindex(columns=FAILURE_CLASS_ORDER, fill_value=0)
    )
    counts['Total failures'] = counts.sum(axis=1)
    for category in FAILURE_CLASS_ORDER:
        counts[f'{category} share'] = counts[category] / counts['Total failures']
    display(Markdown('### Three Initial Solvers: Failure Composition and Residual Failures after Baseline + RG-LNS'))
    display(counts.style.format({f'{c} share': '{:.1%}' for c in FAILURE_CLASS_ORDER}))

    initial_counts = counts.xs('Baseline', level='stage')[FAILURE_CLASS_ORDER]
    initial_shares = initial_counts.div(initial_counts.sum(axis=1), axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
    bottom = np.zeros(len(initial_shares))
    x = np.arange(len(initial_shares))
    for category in FAILURE_CLASS_ORDER:
        values = initial_shares[category].to_numpy()
        axes[0].bar(x, values, bottom=bottom, color=FAILURE_CLASS_COLORS[category], label=category)
        bottom += values
    axes[0].set_xticks(x, initial_shares.index)
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel('Share of all initial failed predictions')
    axes[0].set_title('Mutually exclusive initial failures (100% per bar)')
    axes[0].legend(frameon=False, fontsize=9)

    failure_totals = counts['Total failures'].unstack('stage').reindex(columns=['Baseline', 'Baseline + RG-LNS'])
    failure_totals.plot(kind='bar', ax=axes[1], color=['#495057', '#20c997'], width=0.72)
    axes[1].set_title('Failed predictions before and after RG-LNS')
    axes[1].set_xlabel('Initial solver'); axes[1].set_ylabel('Failed predictions')
    axes[1].tick_params(axis='x', rotation=0)
    axes[1].legend(frameon=False)
    fig.tight_layout()
    display(fig); plt.close(fig)

    repair_summary = (
        classified_outcomes.groupby(['initial_solver', 'failure_type'], observed=False)
        .agg(initial_failures=('repaired_to_perfect', 'size'), repaired_perfect_layouts=('repaired_to_perfect', 'sum'))
    )
    repair_summary['Repair rate'] = repair_summary['repaired_perfect_layouts'] / repair_summary['initial_failures']
    display(Markdown('### Baseline + RG-LNS Repair Rates by Initial Failure Type'))
    display(repair_summary.style.format({'Repair rate': '{:.1%}'}))
else:
    classified_initial = pd.DataFrame()
    classified_ours = pd.DataFrame()
    classified_outcomes = pd.DataFrame()
    print('Quick-check/single-sample mode: skipping population-level proportions and repair rates that require all failures.')

### Observation 1: Consistent Internal Structure at Incorrect Global Positions

Translation consistency of 1.0 means that every piece in a component can reach its true position through the same translation; `coherent_misplacement` additionally requires a nonzero translation. The following cells report sample-level and component-level statistics by initial solver, followed by qualitative examples for `VISUAL_SOLVER` only. The rigid-misplacement-dominated failure category also requires these components to cover at least 40% of the misplaced pieces, a stricter criterion than merely containing one rigidly misplaced component.

In [ ]:
sample_summary = (
    samples.groupby(['initial_solver', 'task'])
    .agg(
        analyzed_puzzle_samples=('lsej_id', 'size'),
        puzzle_samples_with_reliable_component=('has_reliable_component', 'sum'),
        puzzle_samples_with_rigid_misplacement=('has_coherent_misplacement', 'sum'),
        median_largest_component_size=('largest_component_size', 'median'),
    )
)
sample_summary['rate_with_reliable_component'] = (
    sample_summary['puzzle_samples_with_reliable_component'] / sample_summary['analyzed_puzzle_samples']
)
sample_summary['rate_with_rigid_misplacement'] = (
    sample_summary['puzzle_samples_with_rigid_misplacement'] / sample_summary['analyzed_puzzle_samples']
)
display(sample_summary.style.format({
    'rate_with_reliable_component': '{:.1%}', 'rate_with_rigid_misplacement': '{:.1%}',
    'median_largest_component_size': '{:.1f}',
}))

if components.empty:
    component_summary = pd.DataFrame()
    print('No reliable components meet the minimum size requirement.')
else:
    component_summary = (
        components.groupby(['initial_solver', 'task'])
        .agg(
            components=('component_index', 'size'), median_size=('size', 'median'),
            mean_translation_consistency=('translation_consistency', 'mean'),
            median_internal_edge_accuracy=('internal_edge_accuracy', 'median'),
            perfect_rigid_translation_rate=('perfect_rigid_translation', 'mean'),
            coherent_misplacement_rate=('coherent_misplacement', 'mean'),
        )
    )
    display(component_summary.style.format({
        'median_size': '{:.1f}', 'mean_translation_consistency': '{:.1%}',
        'median_internal_edge_accuracy': '{:.1%}', 'perfect_rigid_translation_rate': '{:.1%}',
        'coherent_misplacement_rate': '{:.1%}',
    }))

rigid_presence = sample_summary['rate_with_rigid_misplacement'].unstack('initial_solver')
if not rigid_presence.empty:
    solver_colors = {'Gallagher': '#1f77b4', 'Pomeranz': '#ff7f0e', 'LP': '#2ca02c'}
    axis = rigid_presence.plot(kind='bar', figsize=(8.8, 3.8), width=0.72,
                               color=[solver_colors[solver] for solver in rigid_presence.columns])
    axis.set(ylabel='Share of complete failures\nwith rigidly misplaced components', xlabel='Task', ylim=(0, 1))
    axis.tick_params(axis='x', rotation=0)
    axis.legend(title='Initial solver', frameon=False)
    plt.tight_layout(); display(axis.figure); plt.close(axis.figure)

#### Qualitative Examples for Observation 1

Examples use the largest component in rigid-misplacement-dominated samples from `VISUAL_SOLVER`. The four columns show the shuffled input, baseline result, ground truth, and RG-LNS result. Quantitative comparisons cover all three solvers, while paper examples remain focused.

In [ ]:
def solver_result_title(solver: str) -> str:
    return f'Baseline\n(ViT-T + {solver})'

# Four-column comparison: shuffled input, selected initial solver with AA correctness boxes, ground truth, and Baseline + RG-LNS.
NON_COMPONENT_DIM_ALPHA = 0  # Gently dim non-component regions to preserve image detail.

def draw_component_on_puzzle(
    axis, pieces, positions, grid, title, component_pieces=None, aa_target=None,
    dim_alpha=0.0, show_piece_grid=False,
):
    layout = positions_to_layout(positions, grid)
    component_pieces = set() if component_pieces is None else set(component_pieces)
    component_positions = {divmod(int(positions[piece]), grid) for piece in component_pieces}
    for row in range(grid):
        for column in range(grid):
            piece = int(layout[row, column])
            y = grid - row - 1
            axis.imshow(
                pieces[piece], extent=(column, column + 1, y, y + 1),
                origin='upper', interpolation='nearest',
            )
            if show_piece_grid:
                axis.add_patch(Rectangle(
                    (column, y), 1, 1, fill=False, edgecolor='white', linewidth=0.55, alpha=0.8, zorder=2
                ))
            if component_pieces and piece not in component_pieces and dim_alpha > 0:
                axis.add_patch(Rectangle(
                    (column, y), 1, 1, facecolor='white', edgecolor='none', alpha=dim_alpha, zorder=2
                ))
            if aa_target is not None:
                aa_color = '#2fb344' if int(positions[piece]) == int(aa_target[piece]) else '#e03131'
                axis.add_patch(Rectangle(
                    (column + 0.035, y + 0.035), 0.93, 0.93, fill=False,
                    edgecolor=aa_color, linewidth=1.0, zorder=5,
                ))

    # Draw only the component outline; omit internal piece gridlines to preserve visual coherence.
    boundary_color = '#00b4d8'
    for row, column in component_positions:
        y = grid - row - 1
        if (row - 1, column) not in component_positions:
            axis.plot([column, column + 1], [y + 1, y + 1], color=boundary_color, linewidth=5.0, zorder=7)
        if (row + 1, column) not in component_positions:
            axis.plot([column, column + 1], [y, y], color=boundary_color, linewidth=5.0, zorder=7)
        if (row, column - 1) not in component_positions:
            axis.plot([column, column], [y, y + 1], color=boundary_color, linewidth=5.0, zorder=7)
        if (row, column + 1) not in component_positions:
            axis.plot([column + 1, column + 1], [y, y + 1], color=boundary_color, linewidth=5.0, zorder=7)
    axis.set(xlim=(0, grid), ylim=(0, grid), aspect='equal')
    axis.set_title(title, fontsize=14, fontweight='semibold', pad=12)
    axis.set_xticks([]); axis.set_yticks([])
    for spine in axis.spines.values():
        spine.set_visible(False)

def plot_largest_rigid_component(component_row: pd.Series):
    initial_solver = str(component_row['initial_solver'])
    task = str(component_row['task'])
    key = (initial_solver, task)
    saved_row = int(component_row['saved_row'])
    result_dir = RG_LNS_RESULT_DIRS[key]
    metadata = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
    prediction = RG_LNS_BASELINE_PREDICTIONS[key][saved_row].copy()
    ours_prediction = RG_LNS_SELECTED_PREDICTIONS[key][saved_row].copy()
    target = RG_LNS_TARGETS[key][saved_row].copy()
    grid = int(round(np.sqrt(prediction.size)))
    dataset_index = int(RG_LNS_SAMPLE_INDICES[key][saved_row])
    archive_lsej_id = str(RG_LNS_LSEJ_IDS[key][saved_row])
    if archive_lsej_id != str(component_row['lsej_id']):
        raise RuntimeError('The component record does not match the sample in predictions.npz')

    dataset_class = load_official_dataset_class(LOCAL_DATA_ROOTS[task])
    dataset = dataset_class(LOCAL_DATA_ROOTS[task], split=metadata['split'], task=task, normalize=False)
    sample = dataset[dataset_index]
    pieces = list(chw_uint8_to_hwc_numpy(sample['pieces']))
    component_pieces = {int(piece) for piece in str(component_row['piece_ids']).split(',')}

    shuffled_positions = np.arange(grid * grid, dtype=np.int64)
    aa_correct = int(np.count_nonzero(prediction == target))
    row_shift = int(component_row['dominant_row_shift'])
    column_shift = int(component_row['dominant_column_shift'])
    dim_note = (
        'Non-component pieces are not dimmed.' if NON_COMPONENT_DIM_ALPHA <= 0
        else f'Non-component pieces are gently dimmed with opacity {NON_COMPONENT_DIM_ALPHA:.0%}.'
    )
    display(Markdown(
        f"In sample **{component_row['lsej_id']}** ({task}), the largest rigidly misplaced component contains "
        f"**{int(component_row['size'])}** pieces. The translation from solver positions to true positions is "
        f"$\\Delta=({row_shift}, {column_shift})$. The four columns show the shuffled input, the {initial_solver} baseline, "
        f"ground truth, and Baseline + RG-LNS. Thick solid cyan outlines in the {initial_solver} result and ground truth "
        f"mark the same largest rigidly misplaced component; its outline is omitted from Baseline + RG-LNS. "
        f"Thin green/red boxes in the solver result "
        f"indicate correct/incorrect absolute positions, respectively (AA = {aa_correct}/{grid * grid}). {dim_note}"
    ))
    fig, axes = plt.subplots(
        1, 4, figsize=(15.0, 4.1), constrained_layout=False, gridspec_kw={'wspace': 0.025}
    )
    draw_component_on_puzzle(
        axes[0], pieces, shuffled_positions, grid, 'Shuffled Input', show_piece_grid=True
    )
    draw_component_on_puzzle(
        axes[1], pieces, prediction, grid, solver_result_title(initial_solver),
        component_pieces=component_pieces, aa_target=target, dim_alpha=NON_COMPONENT_DIM_ALPHA,
    )
    draw_component_on_puzzle(
        axes[2], pieces, target, grid, 'Ground Truth',
        component_pieces=component_pieces, dim_alpha=NON_COMPONENT_DIM_ALPHA,
    )
    draw_component_on_puzzle(
        axes[3], pieces, ours_prediction, grid, 'Baseline + RG-LNS',
        aa_target=target, dim_alpha=NON_COMPONENT_DIM_ALPHA,
    )
    fig.subplots_adjust(left=0.002, right=0.998, bottom=0.002, top=0.90, wspace=0.025)
    display(fig)
    plt.close(fig)

# Draw reliability maps from scores and predicted permutations only; blue cells form the same rigidly misplaced component.
def plot_reliability_map(edges_df: pd.DataFrame, component_row: pd.Series, grid: int = 10):
    task, lsej_id = component_row['task'], component_row['lsej_id']
    initial_solver = str(component_row['initial_solver'])
    sample_edges = edges_df[(edges_df['initial_solver'] == initial_solver) & (edges_df['task'] == task) & (edges_df['lsej_id'] == lsej_id)]
    component_pieces = {int(piece) for piece in str(component_row['piece_ids']).split(',')}
    piece_positions = {}
    for edge in sample_edges.itertuples():
        piece_positions[int(edge.anchor)] = (int(edge.row), int(edge.column))
        dr, dc = DIRECTION_OFFSETS[int(edge.direction)]
        piece_positions[int(edge.candidate)] = (int(edge.row) + dr, int(edge.column) + dc)
    fig, axis = plt.subplots(figsize=(7, 7))
    for piece in component_pieces:
        row, column = piece_positions[piece]
        axis.add_patch(Rectangle((column, grid - 1 - row), 1, 1, facecolor='#d0ebff', edgecolor='#1971c2', linewidth=1.5))
    for edge in sample_edges.itertuples():
        color = BUCKET_COLORS[BUCKET_ORDER.index(edge.reliability_bucket)]
        row, column = int(edge.row), int(edge.column)
        if int(edge.direction) == RIGHT:
            x, y = column + 1, grid - row - 0.5
            axis.plot([x, x], [y - 0.35, y + 0.35], color=color, linewidth=3)
        else:
            x, y = column + 0.5, grid - row - 1
            axis.plot([x - 0.35, x + 0.35], [y, y], color=color, linewidth=3)
    axis.set(xlim=(0, grid), ylim=(0, grid), aspect='equal', title=f'{task} | {lsej_id} | Adjacency reliability map')
    axis.set_xticks(range(grid + 1)); axis.set_yticks(range(grid + 1)); axis.grid(color='#dee2e6', linewidth=0.7)
    axis.tick_params(labelbottom=False, labelleft=False)
    display(fig)
    plt.close(fig)

rigid_visual_samples = failure_samples[
    (failure_samples['initial_solver'] == VISUAL_SOLVER)
    & (failure_samples['failure_type'].astype('object') == 'Rigid-misplacement-dominated')
][['initial_solver', 'task', 'lsej_id']]
if components.empty or 'coherent_misplacement' not in components.columns or rigid_visual_samples.empty:
    exemplars = pd.DataFrame()
else:
    exemplars = (
        components[(components['initial_solver'] == VISUAL_SOLVER) & components['coherent_misplacement'].astype(bool)]
        .merge(rigid_visual_samples, on=['initial_solver', 'task', 'lsej_id'], how='inner', validate='many_to_one')
        .sort_values('size', ascending=False)
    )
if len(exemplars):
    exemplar = exemplars.iloc[0]
    exemplar_labels = {
        'initial_solver': 'Initial solver', 'task': 'Task', 'lsej_id': 'LSEJ ID', 'saved_row': 'Result row',
        'component_index': 'Component index', 'size': 'Component size', 'piece_ids': 'Piece IDs',
        'translation_consistency': 'Translation consistency',
        'dominant_row_shift': 'Dominant row shift', 'dominant_column_shift': 'Dominant column shift',
        'nonzero_dominant_shift': 'Nonzero dominant shift',
        'perfect_rigid_translation': 'Perfect rigid translation',
        'coherent_misplacement': 'Rigidly misplaced component',
        'absolute_piece_accuracy': 'Absolute piece accuracy',
        'internal_edges': 'Internal edges', 'internal_edge_accuracy': 'Internal edge accuracy',
        'supported_internal_edges': 'Cycle-supported internal edges', 'supported_edge_accuracy': 'Cycle-supported edge accuracy',
    }
    display(exemplar.rename(index=exemplar_labels).to_frame('Value'))
    display(Markdown('#### Four-Column Comparison of the Largest Rigidly Misplaced Component'))
    plot_largest_rigid_component(exemplar)
    display(Markdown('#### Adjacency Reliability Map of the Same Component'))
    plot_reliability_map(edges, exemplar)
else:
    if selected_filename and not failure_samples.empty:
        selected_type = str(failure_samples.iloc[0]['failure_type'])
        print(f'Skipping Observation 1 visualization: the selected sample is {selected_type}, not rigid-misplacement-dominated.')
    else:
        print('The current subset contains no rigid-misplacement-dominated samples for Observation 1.')


### Observation 2: Are Errors Concentrated in Low-Reliability Regions?

The error rate within a reliability level and the distribution of all errors across reliability levels answer different questions. For all three initial solvers, we report $P(\mathrm{error}\mid\mathrm{level})$ and $P(\mathrm{level}\mid\mathrm{error})$. Levels without complete cycle support are grouped as low-reliability regions. The sample-level failure taxonomy further tests whether these regions account for at least 40% of the misplaced pieces.

In [ ]:
edge_summary = (
    edges.assign(error=~edges['correct'].astype(bool))
    .groupby(['initial_solver', 'task', 'reliability_bucket'], observed=False)
    .agg(edges=('error', 'size'), incorrect_edges=('error', 'sum'))
    .reset_index()
)
edge_summary['error_rate'] = edge_summary['incorrect_edges'] / edge_summary['edges']
incorrect_totals = edge_summary.groupby(['initial_solver', 'task'])['incorrect_edges'].transform('sum')
edge_summary['share_of_all_incorrect_edges'] = np.divide(
    edge_summary['incorrect_edges'], incorrect_totals,
    out=np.zeros(len(edge_summary), dtype=float), where=incorrect_totals.to_numpy() > 0,
)
edge_summary['reliability_bucket'] = pd.Categorical(
    edge_summary['reliability_bucket'], categories=BUCKET_ORDER, ordered=True
)
display(edge_summary.sort_values(['initial_solver', 'task', 'reliability_bucket']).style.format({
    'error_rate': '{:.1%}', 'share_of_all_incorrect_edges': '{:.1%}',
}))

plot_data = edge_summary.groupby(['initial_solver', 'reliability_bucket'], observed=False).agg(
    edges=('edges', 'sum'), incorrect_edges=('incorrect_edges', 'sum')
).reset_index()
plot_data['error_rate'] = plot_data['incorrect_edges'] / plot_data['edges']
solver_error_totals = plot_data.groupby('initial_solver')['incorrect_edges'].transform('sum')
plot_data['error_share'] = np.divide(
    plot_data['incorrect_edges'], solver_error_totals,
    out=np.zeros(len(plot_data), dtype=float), where=solver_error_totals.to_numpy() > 0,
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
width = 0.24
x = np.arange(len(BUCKET_ORDER))
for index, solver in enumerate(INITIAL_SOLVERS):
    solver_rows = plot_data[plot_data['initial_solver'] == solver].set_index('reliability_bucket').reindex(BUCKET_ORDER)
    offset = (index - (len(INITIAL_SOLVERS) - 1) / 2) * width
    axes[0].bar(x + offset, solver_rows['error_rate'], width=width, label=solver)
    axes[1].bar(x + offset, solver_rows['error_share'], width=width, label=solver)
axes[0].set_title('Error rate by reliability level')
axes[0].set_ylabel(r'$P(\mathrm{error}\mid\mathrm{level})$')
axes[1].set_title('Reliability composition of incorrect adjacencies')
axes[1].set_ylabel(r'$P(\mathrm{level}\mid\mathrm{error})$')
for axis in axes:
    axis.set_xticks(x, [BUCKET_LABELS[b] for b in BUCKET_ORDER], rotation=20, ha='right')
    axis.set_ylim(0, 1)
    axis.legend(frameon=False)
fig.tight_layout(); display(fig); plt.close(fig)

#### Qualitative Examples for Observation 2

Examples are selected only from low-reliability-region-error-dominated samples produced by `VISUAL_SOLVER`. The four columns show the baseline result, five-level reliability map, ground truth, and RG-LNS result. A continuous heatmap is retained as a supplementary view.

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

OBSERVATION_TWO_LEGEND_LABELS = [
    'Mutual Top-1 + Cycle',
    f'Mutual Top-{MUTUAL_TOP_K} + Cycle',
    f'Mutual Top-{MUTUAL_TOP_K}, No Cycle',
    f'One-way Top-{MUTUAL_TOP_K}',
    f'Neither Direction in Top-{MUTUAL_TOP_K}',
]

def draw_reliability_panel(axis, sample_edges: pd.DataFrame, prediction: np.ndarray, reliable_pieces: set[int], grid: int):
    layout = positions_to_layout(prediction, grid)
    for row in range(grid):
        for column in range(grid):
            piece = int(layout[row, column])
            y = grid - row - 1
            facecolor = '#d0ebff' if piece in reliable_pieces else '#f1f3f5'
            axis.add_patch(Rectangle(
                (column, y), 1, 1, facecolor=facecolor,
                edgecolor='#ced4da', linewidth=0.45, zorder=1,
            ))

    for edge in sample_edges.itertuples():
        color = BUCKET_COLORS[BUCKET_ORDER.index(edge.reliability_bucket)]
        row, column = int(edge.row), int(edge.column)
        if int(edge.direction) == RIGHT:
            x, y = column + 1, grid - row - 0.5
            axis.plot([x, x], [y - 0.38, y + 0.38], color=color, linewidth=3.6, zorder=3)
        else:
            x, y = column + 0.5, grid - row - 1
            axis.plot([x - 0.38, x + 0.38], [y, y], color=color, linewidth=3.6, zorder=3)

    axis.set(xlim=(0, grid), ylim=(0, grid), aspect='equal')
    axis.set_title('Reliability Map', fontsize=17, fontweight='semibold', pad=12)
    axis.set_xticks([]); axis.set_yticks([])
    for spine in axis.spines.values():
        spine.set_visible(False)


def draw_continuous_confidence_panel(axis, sample_edges: pd.DataFrame, grid: int):
    # metric_postprocess_scores normalizes each query and symmetrizes both directions: lower scores mean higher compatibility.
    confidences = np.clip(1.0 - sample_edges['score'].to_numpy(dtype=float), 0.0, 1.0)
    scale_low, scale_high = np.quantile(confidences, [0.05, 0.95])
    if scale_high - scale_low <= 1e-8:
        scale_low, scale_high = float(confidences.min()), float(confidences.max())
    if scale_high - scale_low <= 1e-8:
        scale_low = max(0.0, float(scale_low) - 0.01)
        scale_high = min(1.0, float(scale_high) + 0.01)
    normalization = Normalize(vmin=scale_low, vmax=scale_high, clip=True)
    colormap = plt.get_cmap('RdYlGn')
    for row in range(grid):
        for column in range(grid):
            y = grid - row - 1
            axis.add_patch(Rectangle(
                (column, y), 1, 1, facecolor='#f8f9fa',
                edgecolor='#dee2e6', linewidth=0.45, zorder=1,
            ))

    for edge in sample_edges.itertuples():
        confidence = float(np.clip(1.0 - float(edge.score), 0.0, 1.0))
        color = colormap(normalization(confidence))
        row, column = int(edge.row), int(edge.column)
        if int(edge.direction) == RIGHT:
            x, y = column + 1, grid - row - 0.5
            axis.plot([x, x], [y - 0.38, y + 0.38], color=color, linewidth=3.6, zorder=3)
        else:
            x, y = column + 0.5, grid - row - 1
            axis.plot([x - 0.38, x + 0.38], [y, y], color=color, linewidth=3.6, zorder=3)

    axis.set(xlim=(0, grid), ylim=(0, grid), aspect='equal')
    axis.set_title('Relative Confidence', fontsize=17, fontweight='semibold', pad=12)
    axis.set_xticks([]); axis.set_yticks([])
    for spine in axis.spines.values():
        spine.set_visible(False)
    return ScalarMappable(norm=normalization, cmap=colormap), (scale_low, scale_high)


def plot_observation_two_example(sample_row: pd.Series):
    initial_solver = str(sample_row['initial_solver'])
    task = str(sample_row['task'])
    key = (initial_solver, task)
    saved_row = int(sample_row['saved_row'])
    result_dir = RG_LNS_RESULT_DIRS[key]
    metadata = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
    prediction = RG_LNS_BASELINE_PREDICTIONS[key][saved_row].copy()
    ours_prediction = RG_LNS_SELECTED_PREDICTIONS[key][saved_row].copy()
    target = RG_LNS_TARGETS[key][saved_row].copy()
    grid = int(round(np.sqrt(prediction.size)))
    dataset_index = int(RG_LNS_SAMPLE_INDICES[key][saved_row])
    archive_lsej_id = str(RG_LNS_LSEJ_IDS[key][saved_row])
    if archive_lsej_id != str(sample_row['lsej_id']):
        raise RuntimeError('The LSEJ ID in the sample record does not match predictions.npz')

    dataset_class = load_official_dataset_class(LOCAL_DATA_ROOTS[task])
    dataset = dataset_class(LOCAL_DATA_ROOTS[task], split=metadata['split'], task=task, normalize=False)
    sample = dataset[dataset_index]
    pieces = list(chw_uint8_to_hwc_numpy(sample['pieces']))
    sample_edges = edges[
        (edges['initial_solver'] == initial_solver) & (edges['task'] == task) & (edges['lsej_id'] == str(sample_row['lsej_id']))
    ]
    sample_components = components[
        (components['initial_solver'] == initial_solver) & (components['task'] == task) & (components['lsej_id'] == str(sample_row['lsej_id']))
    ] if not components.empty else pd.DataFrame()
    reliable_pieces = set()
    if not sample_components.empty:
        for piece_ids in sample_components['piece_ids'].astype(str):
            reliable_pieces.update(int(piece) for piece in piece_ids.split(',') if piece)

    display(Markdown(
        f"Sample **{sample_row['lsej_id']}** ({task}) contains **{int(sample_row['wrong_pieces'])}** "
        f"pieces at incorrect absolute positions, **{float(sample_row['low_reliability_wrong_piece_ratio']):.1%}** of which lie in "
        f"low-reliability regions; **{float(sample_row['low_reliability_incorrect_adjacency_ratio']):.1%}** of incorrect adjacencies "
        f"belong to levels without cycle support. Thin red/green boxes in column 1 indicate incorrect/correct positions. In column 2, light-blue cells indicate cycle-supported "
        f"reliable components, gray cells indicate low-reliability regions, and colored segments represent the five adjacency reliability levels. "
        f"Column 4 shows the final RG-LNS result."
    ))

    fig, axes = plt.subplots(
        1, 4, figsize=(15.0, 4.73), constrained_layout=False,
        gridspec_kw={'wspace': 0.025},
    )
    draw_component_on_puzzle(
        axes[0], pieces, prediction, grid, solver_result_title(initial_solver), aa_target=target
    )
    draw_reliability_panel(axes[1], sample_edges, prediction, reliable_pieces, grid)
    draw_component_on_puzzle(axes[2], pieces, target, grid, 'Ground Truth')
    draw_component_on_puzzle(axes[3], pieces, ours_prediction, grid, 'Baseline + RG-LNS', aa_target=target)

    legend_handles = [
        Line2D([0], [0], color=color, linewidth=4.5, label=label)
        for label, color in zip(OBSERVATION_TWO_LEGEND_LABELS, BUCKET_COLORS)
    ]
    fig.legend(
        handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.995),
        ncol=5, frameon=False, fontsize=12, columnspacing=1.0, handlelength=1.8,
    )
    fig.subplots_adjust(left=0.002, right=0.998, bottom=0.002, top=0.84, wspace=0.025)
    display(fig)
    plt.close(fig)

    display(Markdown(
        '**Within-puzzle relative confidence heatmap.** Column 2 uses '
        '$\\mathrm{confidence}=1-\\mathrm{score}$ and maps the 5th/95th percentiles of current adjacency confidence values '
        'to red/green, respectively. Colors indicate relative confidence within this puzzle.'
    ))
    continuous_fig, continuous_axes = plt.subplots(
        1, 4, figsize=(15.0, 4.73), constrained_layout=False,
        gridspec_kw={'wspace': 0.025},
    )
    draw_component_on_puzzle(
        continuous_axes[0], pieces, prediction, grid,
        solver_result_title(initial_solver), aa_target=target,
    )
    confidence_mappable, confidence_scale = draw_continuous_confidence_panel(
        continuous_axes[1], sample_edges, grid
    )
    draw_component_on_puzzle(
        continuous_axes[2], pieces, target, grid, 'Ground Truth'
    )
    draw_component_on_puzzle(
        continuous_axes[3], pieces, ours_prediction, grid, 'Baseline + RG-LNS', aa_target=target
    )
    colorbar_axis = continuous_axes[1].inset_axes([0.06, 1.18, 0.88, 0.035])
    colorbar = continuous_fig.colorbar(
        confidence_mappable, cax=colorbar_axis, orientation='horizontal'
    )
    scale_low, scale_high = confidence_scale
    colorbar.set_ticks([scale_low, 0.5 * (scale_low + scale_high), scale_high])
    colorbar.set_label('Within-puzzle bidirectional confidence', fontsize=9, labelpad=2)
    colorbar.ax.xaxis.set_ticks_position('top')
    colorbar.ax.xaxis.set_label_position('top')
    colorbar.ax.tick_params(labelsize=8, pad=1)
    continuous_fig.subplots_adjust(
        left=0.002, right=0.998, bottom=0.002, top=0.78, wspace=0.025
    )
    display(continuous_fig)
    plt.close(continuous_fig)


low_reliability_mask = (
    (failure_samples['initial_solver'] == VISUAL_SOLVER)
    & (failure_samples['failure_type'].astype('object') == 'Low-reliability-region-error-dominated')
)
observation_two_candidates = failure_samples[low_reliability_mask].copy()
observation_two_selection_note = (
    'Selected low-reliability-region-error-dominated sample'
    if selected_filename else 'Low-reliability-region-error-dominated sample'
)

if observation_two_candidates.empty:
    if selected_filename and not failure_samples.empty:
        selected_type = str(failure_samples.iloc[0]['failure_type'])
        print(f'Skipping Observation 2 visualization: the selected sample is {selected_type}, not low-reliability-region-error-dominated.')
    else:
        print('The current subset contains no low-reliability-region-error-dominated samples for Observation 2.')
else:
    observation_two_exemplar = observation_two_candidates.sort_values(
        ['low_reliability_wrong_piece_ratio', 'wrong_pieces',
         'low_reliability_incorrect_adjacency_ratio'],
        ascending=[False, False, False],
    ).iloc[0]
    display(Markdown(f'#### Four-Column Qualitative Example for Observation 2 | {observation_two_selection_note}'))
    plot_observation_two_example(observation_two_exemplar)


## Threshold Sensitivity Analysis

The main analysis uses a 40% dominance threshold, with 50%, 70%, and 90% as stricter alternatives. Sensitivity analysis reports the three failure proportions separately for each initial solver to assess whether cross-solver conclusions depend on a single threshold. This table is generated only in full-population mode.

In [ ]:
if not FULL_POPULATION_ANALYSIS:
    print('Quick-check/single-sample mode: skipping population-level threshold sensitivity statistics.')
else:
    sensitivity_frames = []
    for threshold in SENSITIVITY_THRESHOLDS:
        threshold_samples = samples.copy()
        threshold_samples['failure_type'] = threshold_samples.apply(
            assign_failure_type, axis=1, threshold=threshold
        )
        threshold_classified = classify_all_failures(paired_outcomes, threshold_samples, 'Baseline')
        summary = (
            threshold_classified.groupby(['initial_solver', 'task', 'failure_type']).size()
            .rename('count').reset_index()
        )
        totals = summary.groupby(['initial_solver', 'task'])['count'].transform('sum')
        summary['share'] = summary['count'] / totals
        summary['threshold'] = threshold
        sensitivity_frames.append(summary)
    sensitivity = pd.concat(sensitivity_frames, ignore_index=True)
    sensitivity_table = sensitivity.pivot_table(
        index=['initial_solver', 'task', 'threshold'], columns='failure_type',
        values='share', fill_value=0,
    ).reindex(columns=FAILURE_CLASS_ORDER, fill_value=0)
    display(sensitivity_table.style.format('{:.1%}'))

## Sample-Level Audit Details

Full-population mode shows the classification evidence for the first few complete failures from each initial solver for manual inspection. Incomplete predictions lack full-grid adjacency and are included in other/mixed failures without fabricating component or adjacency evidence.

In [ ]:
if not failure_samples.empty:
    sample_failure_preview = failure_samples[[
        'initial_solver', 'task', 'lsej_id', 'wrong_pieces',
        'rigid_wrong_piece_ratio', 'low_reliability_wrong_piece_ratio',
        'low_reliability_incorrect_adjacency_ratio', 'failure_type',
    ]].groupby(['initial_solver', 'task'], sort=False).head(20).rename(columns={
        'initial_solver': 'Initial solver', 'task': 'Task', 'lsej_id': 'Sample name',
        'wrong_pieces': 'Misplaced pieces', 'rigid_wrong_piece_ratio': 'Rigid-misplacement wrong-piece ratio',
        'low_reliability_wrong_piece_ratio': 'Low-reliability wrong-piece ratio',
        'low_reliability_incorrect_adjacency_ratio': 'Low-reliability share of incorrect adjacencies',
        'failure_type': 'Failure type',
    })
    display(sample_failure_preview.style.format({
        'Rigid-misplacement wrong-piece ratio': '{:.1%}', 'Low-reliability wrong-piece ratio': '{:.1%}',
        'Low-reliability share of incorrect adjacencies': '{:.1%}',
    }))
else:
    print('No complete failed samples are available to display.')

## Guidance for Use in the Paper

- Formal tables must use full-population results with `RG_LNS_ANALYSIS_MAX_FAILED=0`, no selected individual file, and a successful strict protocol audit.
- The initial failure composition table tests whether the three solvers share similar failure structures; the three category proportions sum to 100% for each solver.
- Repair rates by category assess whether RG-LNS addresses rigid misplacement and low-reliability regions, beyond improving overall PA.
- Observation 1 combines the proportion of failed samples containing rigidly misplaced components, component translation consistency, and qualitative figures. Observation 2 combines the two conditional probabilities, the proportion of low-reliability-dominated samples, and qualitative figures.
- These results provide evidence of associations between failure structures and repair outcomes, not strict causal conclusions.
- `VISUAL_SOLVER` affects only qualitative figures, not quantitative statistics for the three solvers.